In [ ]:
# =============================================================
# TAHAP 5: MODEL EVALUATION
# Mata Kuliah Penalaran Komputer - CBR Korupsi Kerugian Keuangan Negara
# =============================================================

# Tahap 5: Model Evaluation
**Tujuan:** Mengukur dan menganalisis performa retrieval & prediksi
menggunakan Accuracy, Precision, Recall, dan F1-score.

In [ ]:
Import Library
import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
from typing import List, Dict
from collections import Counter

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.model_selection import cross_val_score
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
Konfigurasi Path
BASE_DIR  = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) \
            if "__file__" in dir() else os.getcwd()
PROC_DIR  = os.path.join(BASE_DIR, "data", "processed")
EVAL_DIR  = os.path.join(BASE_DIR, "data", "eval")
RES_DIR   = os.path.join(BASE_DIR, "data", "results")
MODEL_DIR = os.path.join(BASE_DIR, "models")
RAW_DIR   = os.path.join(BASE_DIR, "data", "raw")
FIG_DIR   = os.path.join(BASE_DIR, "data", "eval", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
sys.path.insert(0, os.path.join(BASE_DIR, "notebooks"))
print("✅ Path siap")

In [ ]:
Load Semua Artefak
def load_artifacts():
    """Load dataset, model TF-IDF, solusi kasus, dan query evaluasi."""
    # Dataset
    csv_path = os.path.join(PROC_DIR, "cases_full.csv")
    if not os.path.exists(csv_path):
        csv_path = os.path.join(PROC_DIR, "cases.csv")
    df = pd.read_csv(csv_path, encoding="utf-8-sig")

    # TF-IDF
    with open(os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"), "rb") as f:
        vectorizer = pickle.load(f)
    tfidf_mat = np.load(os.path.join(MODEL_DIR, "tfidf_matrix.npy"))
    from scipy.sparse import csr_matrix
    tfidf_mat = csr_matrix(tfidf_mat)
    with open(os.path.join(MODEL_DIR, "case_ids.json"), "r") as f:
        case_ids = json.load(f)

    # Case Solutions
    sol_path = os.path.join(MODEL_DIR, "case_solutions.json")
    with open(sol_path, "r", encoding="utf-8") as f:
        case_solutions = json.load(f)

    # Query evaluasi
    q_path = os.path.join(EVAL_DIR, "queries.json")
    with open(q_path, "r", encoding="utf-8") as f:
        queries = json.load(f)

    print(f"✅ Artefak loaded | {len(df)} kasus | {len(queries)} query evaluasi")
    return df, vectorizer, tfidf_mat, case_ids, case_solutions, queries

In [ ]:
Fungsi Retrieval (Ulang dari Tahap 3)
from sklearn.metrics.pairwise import cosine_similarity

def preprocess_text(text: str) -> str:
    import re
    stopwords_id = {
        "yang","dan","di","ke","dari","ini","itu","dengan","untuk","pada",
        "adalah","dalam","tidak","oleh","atau","juga","sudah","akan","ada",
        "bahwa","telah","tersebut","lebih","setelah","serta","dapat","harus",
        "namun","karena","atas","tapi","jika","maka","sebagai","antara","hal",
    }
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if t not in stopwords_id and len(t) > 2]
    return " ".join(tokens)

def retrieve(query: str, k: int, vectorizer, tfidf_matrix, case_ids):
    proc = preprocess_text(query)
    qvec = vectorizer.transform([proc])
    sims = cosine_similarity(qvec, tfidf_matrix).flatten()
    top  = sims.argsort()[::-1][:k]
    return [(case_ids[i], float(sims[i])) for i in top]

def predict_label(query: str, k: int, vectorizer, tfidf_matrix,
                  case_ids, case_solutions, method="weighted"):
    results = retrieve(query, k, vectorizer, tfidf_matrix, case_ids)
    ids     = [r[0] for r in results]
    scores  = [r[1] for r in results]
    sols    = [case_solutions[i] for i in ids if i in case_solutions]
    if not sols:
        return "bersalah"
    if method == "majority":
        labels = [s["label"] for s in sols]
        return Counter(labels).most_common(1)[0][0]
    weight_map = {}
    for s, sc in zip(sols, scores):
        lbl = s["label"]
        weight_map[lbl] = weight_map.get(lbl, 0) + sc
    return max(weight_map, key=weight_map.get)

In [ ]:
Evaluasi Retrieval (Recall@k, Hit Rate)
def eval_retrieval(queries: List[Dict], vectorizer, tfidf_matrix, case_ids,
                   k_list: List[int] = [1, 3, 5, 10]) -> pd.DataFrame:
    """
    Evaluasi fungsi retrieval:
    - Recall@k : apakah ground-truth ada di top-k?
    - MRR (Mean Reciprocal Rank)
    """
    results = []
    for k in k_list:
        hits, mrr_sum = 0, 0.0
        for q in queries:
            top_k = retrieve(q["query_text"], k, vectorizer, tfidf_matrix, case_ids)
            top_ids = [r[0] for r in top_k]
            gt = q["ground_truth_id"]
            if gt in top_ids:
                hits    += 1
                rank     = top_ids.index(gt) + 1
                mrr_sum += 1.0 / rank
        recall_at_k = hits / len(queries)
        mrr         = mrr_sum / len(queries)
        results.append({
            "k"          : k,
            "Recall@k"   : round(recall_at_k, 4),
            "MRR"        : round(mrr, 4),
            "Hits"       : hits,
            "Total Query": len(queries),
        })
        print(f"  Recall@{k:<2}: {recall_at_k:.4f} | MRR: {mrr:.4f} | Hits: {hits}/{len(queries)}")
    return pd.DataFrame(results)

In [ ]:
Evaluasi Klasifikasi (Accuracy, Precision, Recall, F1)
def eval_classification(queries: List[Dict], df: pd.DataFrame,
                        vectorizer, tfidf_matrix, case_ids,
                        case_solutions: Dict, k: int = 5) -> Dict:
    """
    Evaluasi prediksi label menggunakan ground-truth label dari dataset.
    Metode: majority vote & weighted similarity.
    """
    y_true, y_mv, y_ws = [], [], []

    for q in queries:
        gt_id    = q["ground_truth_id"]
        gt_row   = df[df["case_id"] == gt_id]
        if gt_row.empty:
            continue
        gt_label = gt_row["label"].values[0]
        y_true.append(gt_label)

        pred_mv = predict_label(q["query_text"], k, vectorizer, tfidf_matrix,
                                 case_ids, case_solutions, method="majority")
        pred_ws = predict_label(q["query_text"], k, vectorizer, tfidf_matrix,
                                 case_ids, case_solutions, method="weighted")
        y_mv.append(pred_mv)
        y_ws.append(pred_ws)

    def compute_metrics(y_t, y_p, name):
        avg = "weighted" if len(set(y_t)) > 2 else "binary"
        pos = list(set(y_t))[0] if len(set(y_t)) == 1 else None
        return {
            "Model"    : name,
            "Accuracy" : round(accuracy_score(y_t, y_p), 4),
            "Precision": round(precision_score(y_t, y_p, average=avg, zero_division=0), 4),
            "Recall"   : round(recall_score(y_t, y_p, average=avg, zero_division=0), 4),
            "F1-Score" : round(f1_score(y_t, y_p, average=avg, zero_division=0), 4),
        }

    mv_metrics = compute_metrics(y_true, y_mv, "CBR Majority Vote")
    ws_metrics = compute_metrics(y_true, y_ws, "CBR Weighted Similarity")

    metrics_df = pd.DataFrame([mv_metrics, ws_metrics])

    print("\n  📊 Classification Report — Majority Vote:")
    print(classification_report(y_true, y_mv, zero_division=0))
    print("  📊 Classification Report — Weighted Similarity:")
    print(classification_report(y_true, y_ws, zero_division=0))

    return {
        "metrics_df": metrics_df,
        "y_true"    : y_true,
        "y_mv"      : y_mv,
        "y_ws"      : y_ws,
    }

In [ ]:
Evaluasi Model SVM & NB dari Tahap 3
def eval_saved_classifiers(df: pd.DataFrame) -> pd.DataFrame:
    """Load dan evaluasi classifier SVM & Naive Bayes yang sudah disimpan."""
    from sklearn.feature_extraction.text import TfidfVectorizer as TV
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder

    csv_full = os.path.join(PROC_DIR, "cases_full.csv")
    if not os.path.exists(csv_full):
        return pd.DataFrame()

    df_full = pd.read_csv(csv_full, encoding="utf-8-sig")
    texts, labels = [], []
    for _, row in df_full.iterrows():
        txt_path = os.path.join(RAW_DIR, f"{row['case_id']}.txt")
        if os.path.exists(txt_path):
            with open(txt_path, "r", encoding="utf-8") as f:
                texts.append(preprocess_text(f.read()))
        elif pd.notna(row.get("text_full", "")):
            texts.append(preprocess_text(str(row.get("text_full", ""))))
        else:
            texts.append(preprocess_text(str(row.get("ringkasan_fakta", ""))))
        labels.append(row["label"])

    le = LabelEncoder()
    y  = le.fit_transform(labels)
    if len(set(y)) < 2:
        print("  ⚠️  Hanya 1 kelas label — evaluasi classifier dilewati.")
        return pd.DataFrame()

    vec = TV(max_features=3000, ngram_range=(1,2), sublinear_tf=True)
    X   = vec.fit_transform(texts)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    records = []
    for mname in ["svm", "naive_bayes"]:
        mpath = os.path.join(MODEL_DIR, f"classifier_{mname}.pkl")
        if not os.path.exists(mpath):
            continue
        with open(mpath, "rb") as f:
            pkg = pickle.load(f)
        clf   = pkg["model"]
        v_cls = pkg["vectorizer"]
        X_te2 = v_cls.transform([preprocess_text(t) for t in
                                   [df_full.iloc[i]["case_id"] for i in range(len(df_full))
                                    if i < len(texts)][:len(y_te)]])
        # Gunakan vector dari fit yang sama
        y_pred = clf.predict(X_te)
        avg    = "weighted"
        records.append({
            "Model"    : mname.upper().replace("_", " "),
            "Accuracy" : round(accuracy_score(y_te, y_pred), 4),
            "Precision": round(precision_score(y_te, y_pred, average=avg, zero_division=0), 4),
            "Recall"   : round(recall_score(y_te, y_pred, average=avg, zero_division=0), 4),
            "F1-Score" : round(f1_score(y_te, y_pred, average=avg, zero_division=0), 4),
        })
        print(f"  {mname.upper()} — Acc: {records[-1]['Accuracy']:.4f} | F1: {records[-1]['F1-Score']:.4f}")

    return pd.DataFrame(records)

In [ ]:
Analisis Kegagalan (Error Analysis)
def error_analysis(queries, y_true, y_pred, label_name="Weighted Sim") -> pd.DataFrame:
    """Identifikasi kasus yang gagal diprediksi dengan benar."""
    errors = []
    for i, (qt, yp) in enumerate(zip(y_true, y_pred)):
        if qt != yp:
            errors.append({
                "query_id"      : queries[i]["query_id"],
                "ground_truth"  : qt,
                "predicted"     : yp,
                "query_snippet" : queries[i]["query_text"][:150] + "...",
            })
    print(f"\n  ❌ Kasus gagal ({label_name}): {len(errors)} dari {len(y_true)}")
    for e in errors:
        print(f"     {e['query_id']} | GT: {e['ground_truth']} → Pred: {e['predicted']}")
        print(f"     '{e['query_snippet'][:100]}'")
    return pd.DataFrame(errors)

In [ ]:
Visualisasi
def plot_metrics(metrics_df: pd.DataFrame, retrieval_df: pd.DataFrame):
    """Membuat bar chart perbandingan metrik model."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        "Evaluasi CBR — Korupsi Kerugian Keuangan Negara",
        fontsize=13, fontweight="bold"
    )

    # Plot 1: Classification Metrics
    if not metrics_df.empty and "Model" in metrics_df.columns:
        ax1    = axes[0]
        metric_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]
        existing = [c for c in metric_cols if c in metrics_df.columns]
        x      = np.arange(len(existing))
        width  = 0.35
        colors = ["#2196F3", "#4CAF50"]
        for i, (_, row) in enumerate(metrics_df.iterrows()):
            vals = [row.get(c, 0) for c in existing]
            bars = ax1.bar(x + i*width, vals, width, label=row["Model"], color=colors[i % len(colors)], alpha=0.85)
            for bar in bars:
                ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                         f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
        ax1.set_xticks(x + width/2)
        ax1.set_xticklabels(existing, rotation=15)
        ax1.set_ylim(0, 1.15)
        ax1.set_ylabel("Skor")
        ax1.set_title("Perbandingan Metrik Klasifikasi")
        ax1.legend(fontsize=8)
        ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        ax1.grid(axis="y", alpha=0.3)

    # Plot 2: Recall@k
    if not retrieval_df.empty:
        ax2 = axes[1]
        ax2.plot(retrieval_df["k"], retrieval_df["Recall@k"],
                 marker="o", color="#E91E63", linewidth=2, markersize=8, label="Recall@k")
        ax2.plot(retrieval_df["k"], retrieval_df["MRR"],
                 marker="s", color="#FF9800", linewidth=2, markersize=8, linestyle="--", label="MRR")
        for _, row in retrieval_df.iterrows():
            ax2.annotate(f"{row['Recall@k']:.3f}", (row["k"], row["Recall@k"]),
                         textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
        ax2.set_xlabel("k (jumlah kasus yang diambil)")
        ax2.set_ylabel("Skor")
        ax2.set_title("Recall@k dan MRR — Retrieval Performance")
        ax2.set_ylim(0, 1.1)
        ax2.legend()
        ax2.grid(alpha=0.3)
        ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, "evaluation_results.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  📊 Grafik tersimpan: {fig_path}")

In [ ]:
Pipeline Evaluasi Utama
def run_evaluation_pipeline():
    print("\n" + "="*60)
    print("  TAHAP 5: MODEL EVALUATION")
    print("="*60)

    # Load
    print("\n[1/5] Load artefak...")
    df, vectorizer, tfidf_mat, case_ids, case_solutions, queries = load_artifacts()

    # Evaluasi Retrieval
    print("\n[2/5] Evaluasi Retrieval (Recall@k & MRR)...")
    retrieval_df = eval_retrieval(queries, vectorizer, tfidf_mat, case_ids)
    retrieval_path = os.path.join(EVAL_DIR, "retrieval_metrics.csv")
    retrieval_df.to_csv(retrieval_path, index=False, encoding="utf-8-sig")
    print(f"\n  ✅ Retrieval metrics tersimpan: {retrieval_path}")
    print(retrieval_df.to_string(index=False))

    # Evaluasi Klasifikasi CBR
    print("\n[3/5] Evaluasi Prediksi Label (CBR Majority Vote vs Weighted)...")
    cls_result = eval_classification(
        queries, df, vectorizer, tfidf_mat, case_ids, case_solutions, k=5
    )
    metrics_df = cls_result["metrics_df"]

    # Evaluasi Classifier SVM & NB
    print("\n[4/5] Evaluasi Classifier SVM & Naive Bayes...")
    clf_df = eval_saved_classifiers(df)
    if not clf_df.empty:
        metrics_df = pd.concat([metrics_df, clf_df], ignore_index=True)

    pred_path = os.path.join(EVAL_DIR, "prediction_metrics.csv")
    metrics_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
    print(f"\n  ✅ Prediction metrics tersimpan: {pred_path}")
    print("\n  📊 Tabel Metrik Lengkap:")
    print(metrics_df.to_string(index=False))

    # Error Analysis
    print("\n[5/5] Analisis Kegagalan Model...")
    err_df = error_analysis(
        queries,
        cls_result["y_true"],
        cls_result["y_ws"],
        label_name="Weighted Similarity"
    )
    if not err_df.empty:
        err_path = os.path.join(EVAL_DIR, "error_analysis.csv")
        err_df.to_csv(err_path, index=False, encoding="utf-8-sig")
        print(f"  Error analysis tersimpan: {err_path}")

    # Rekomendasi Perbaikan
    print("\n" + "="*60)
    print("  REKOMENDASI PERBAIKAN MODEL")
    print("="*60)
    print("""
  1. AUGMENTASI DATA
     → Tambahkan lebih banyak putusan (>100 dokumen) agar distribusi
       label lebih merata dan model belajar pola yang lebih beragam.

  2. PENINGKATAN REPRESENTASI TEKS
     → Gunakan IndoBERT (indobenchmark/indobert-base-p1) untuk
       embedding semantik yang lebih kaya dibanding TF-IDF statistik.
     → Tambahkan fitur metadata: pasal, tahun, nama pengadilan.

  3. OPTIMASI RETRIEVAL
     → Coba BM25 sebagai alternatif TF-IDF untuk retrieval berbasis
       term frequency yang lebih robust.
     → Implementasi re-ranking dengan model cross-encoder.

  4. PERBAIKAN LABEL
     → Perluas kategori label: 'bersalah_berat', 'bersalah_ringan',
       'bebas', untuk granularitas prediksi yang lebih tinggi.

  5. EVALUASI LEBIH ROBUST
     → Gunakan k-fold cross validation pada data yang lebih besar.
     → Tambahkan metrik NDCG dan MAP untuk evaluasi retrieval ranking.
  """)

    # Visualisasi
    plot_metrics(metrics_df, retrieval_df)

    print("✅ Tahap 5 selesai! Semua output tersimpan di /data/eval/")
    print("="*60)

if __name__ == "__main__":
    run_evaluation_pipeline()